# 10장 작은 GPT 조립하기

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 10-1 준비

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import math
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, show

vocab = Vocab.build()
print("글자 사전 크기:", len(vocab))

## 실습 10-2 붕어빵 틀

In [ ]:
class FishBread:
    def __init__(self, filling):
        self.filling = filling

    def describe(self):
        return self.filling + " 붕어빵"

bread1 = FishBread("팥")
bread2 = FishBread("슈크림")
print(bread1.describe())
print(bread2.describe())
print(bread1.filling)

## 실습 10-3 비율 표를 만드는 부품

In [ ]:
torch.manual_seed(0)
emb = nn.Embedding(len(vocab), 8)
chars = list("토끼가 숲에 가요")
ids = torch.tensor(vocab.encode("토끼가 숲에 가요"))
cards = emb(ids)
print("숫자 카드 모양:", cards.shape)

## 실습 10-4 비율 표를 만드는 부품

**먼저 예상해 보세요.** 아홉 글자의 비율 표에서 첫 줄 '토'는 비율을 어떻게 나눠 줄까요?

In [ ]:
class RatioTable(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.head_size = head_size

    def forward(self, x):
        q = self.query(x)                               # ① 질문 쪽지
        k = self.key(x)                                 # ① 명찰
        scores = q @ k.T / self.head_size ** 0.5        # ② 점수, ③ 나누기
        T = x.shape[0]
        mask = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(mask == 0, float("-inf"))   # ④ 가리기
        return F.softmax(scores, dim=-1)                # ⑤ 비율

torch.manual_seed(0)
ratio = RatioTable(8, 4)
W9 = ratio(cards)
print("비율 표 모양:", W9.shape)
show(W9, rows=chars, cols=chars)
print("줄마다 합:", W9.sum(dim=-1))

## 실습 10-5 이야기 데이터 준비

In [ ]:
from modu_llm import load_stories, encode_texts, get_batch

torch.manual_seed(42)
stories = load_stories(200)
data = encode_texts(stories, vocab)
print("글자 수:", len(data))

## 실습 10-6 설정 다섯 개

In [ ]:
from modu_llm import GPTConfig

cfg = GPTConfig()
print("사전 크기:", cfg.vocab_size)
print("문맥 길이:", cfg.block_size)
print("카드 길이:", cfg.n_embd)
print("헤드 수:", cfg.n_head)
print("블록 층 수:", cfg.n_layer)
print("글자 사전과 크기가 같은가요?", cfg.vocab_size == len(vocab))

## 실습 10-7 네 부품

In [ ]:
from modu_llm import Embeddings, Block

class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = Embeddings(cfg)                          # ① 글자 카드 + 자리 카드
        self.blocks = nn.ModuleList()
        for i in range(cfg.n_layer):
            self.blocks.append(Block(cfg))                  # ② 블록 6층
        self.ln = nn.LayerNorm(cfg.n_embd)                  # ③ 마지막 정리
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size)   # ④ 출력층

    def forward(self, idx, targets=None):
        x = self.emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln(x)
        logits = self.head(x)
        if targets is None:
            return logits, None
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

model = TinyGPT(cfg)
print(model.head)

## 실습 10-8 정답 글자에 준 확률로 재기

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0]])   # 문제 1개, 후보 글자 3개의 점수
answer = torch.tensor([0])                  # 정답은 0번 글자

probs = F.softmax(scores, dim=-1)
print("확률:", probs)
print("-log(정답 확률):", -torch.log(probs[0, 0]))
print("cross_entropy:", F.cross_entropy(scores, answer))

## 실습 10-9 확률이 작을수록 커지는 손실

**먼저 예상해 보세요.** 규칙대로라면 확률 0.001의 손실은 얼마일까요?

In [ ]:
for p in [0.5, 0.25, 0.1, 0.01, 0.001, 1 / 2453]:
    print(f"정답 확률 {p:.4f} → 손실 {-math.log(p):.2f}")

## 실습 10-10 문제 여러 개의 손실

In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0],     # 문제 1
                       [0.0, 3.0, 0.0],     # 문제 2
                       [1.0, 1.0, 1.0]])    # 문제 3
answers = torch.tensor([0, 1, 2])           # 문제마다 정답 번호

probs = F.softmax(scores, dim=-1)
show(probs, rows=["문제 1", "문제 2", "문제 3"], cols=["0번", "1번", "2번"])
picked = torch.stack([probs[0, 0], probs[1, 1], probs[2, 2]])   # 정답 글자의 확률
print("정답 확률:", picked)
print("문제마다 손실:", -torch.log(picked))
print("손실 평균:", (-torch.log(picked)).mean())
print("cross_entropy:", F.cross_entropy(scores, answers))

## 실습 10-11 학습 전 모델의 손실

**먼저 예상해 보세요.** 아무것도 배우지 않은 모델의 손실은 얼마쯤일까요?

In [ ]:
x, y = get_batch(data, cfg.block_size, 4)
logits, loss = model(x, y)
print("x 모양:", x.shape)
print("logits 모양:", logits.shape)
print(f"학습 전 손실: {loss.item():.2f}")
print(f"고르게 찍을 때 손실: {-math.log(1 / len(vocab)):.2f}")

## 실습 10-12 학습으로 고쳐 나갈 숫자

**먼저 예상해 보세요.** 임베딩, 블록 6층, 출력층 가운데 파라미터가 가장 많은 곳은 어디일까요?

In [ ]:
def count(m):
    total = 0
    for p in m.parameters():
        total = total + p.numel()
    return total

print(f"임베딩: {count(model.emb):,}")
print(f"블록 6층: {count(model.blocks):,}")
print(f"출력층: {count(model.ln) + count(model.head):,}")
print(f"전체: {count(model):,}")

## 실습 10-13 비포 사진 찍기

**먼저 예상해 보세요.** 학습 전 손실이 고양이 수준이었으니, 어떤 글이 나올까요?

In [ ]:
from modu_llm import generate

text = generate(model, vocab, "어느 날, ", max_new_tokens=80, stop_at_end=False)
print("어느 날, " + text)

## 실습 10-14 더 해 보기: 크기를 바꾸면

**먼저 예상해 보세요.** 블록 층 수를 6에서 12로 두 배 늘리는 것과, 카드 길이를 256에서 512로 두 배 늘리는 것 가운데 어느 쪽이 파라미터가 더 많이 늘까요?

In [ ]:
for n_layer, n_embd in [(6, 256), (12, 256), (6, 512)]:
    m = TinyGPT(GPTConfig(n_layer=n_layer, n_embd=n_embd))
    print(f"층 {n_layer}, 카드 길이 {n_embd}: {count(m):,}")